# Quantum Entanglement for Decision-Making

Quantum mechanics allows us to do things that wouldn't be possible if our world were merely classical.
A quantum computer can factor large numbers far faster than any classical algorithm we know of, which is why quantum computing is a threat to the public-key cryptography that secures the internet today.
Two parties with a quantum channel between them can establish a shared secret key whose security rests on physical law rather than on some computation being too hard to do.

Beyond computing and communication, quantum mechanics has still more tricks up its sleeve.
In 1964 the physicist John Bell showed that quantum entanglement can be an aid to decision-making.
Specifically, it can help cooperating agents coordinate their actions when communication between them is restricted.

Consider a situation of the following kind.
Two agents are cooperating on some task.
They have to act at the same moment, in different places, with no time to talk to each other.
Each one sees part of what is going on, but not the part the other sees.
How well they do depends on both of their actions together.
They can agree on a plan in advance, but once the moment arrives each of them is on their own.

It is natural to assume that a carefully chosen plan, together with a stock of shared random numbers, is the best two such agents can possibly do.
What Bell showed is that this assumption is false.
If the agents also share entangled quantum particles, and each measures their own particles before acting, they can produce patterns of joint behaviour that no advance plan can reproduce, however clever the plan and however much shared randomness goes into it.
This has been confirmed with increasingly careful experimental work over the ensuing decades (some of which was the subject of the 2022 Nobel Prize in Physics).

It is worth being clear about what entanglement does and does not do here.
It does not let the agents send each other messages, and neither one learns anything about what the other saw.
What it gives them is a better correlation resource than shared randomness, and a better correlation resource is exactly what two agents need when their decisions have to fit together but they cannot talk.

In this challenge you will be given cooperative two-player games and asked to find entangled strategies that strictly beat every strategy available without entanglement.



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from check_validity import is_valid_povm, is_valid_state
from games import (
    TOLERANCE,
    get_classical_best,
    classical_values,
    expected_payoff,
    fingerprint,
    get_game,
    save_witnesses,
    score,
)





# Warmup: the CHSH Game

## Setup
Alice and Bob play the following game, called [the CHSH game](https://en.wikipedia.org/wiki/CHSH_inequality#CHSH_game):
A referee randomly draws two bits $x,y\in\{0,1\}$ uniformly and independently and gives $x$ to Alice and $y$ to Bob.
Alice and Bob keep their bits secret from each other.
Once they have received their secret bits, Alice and Bob each choose a bit, to send back to the referee, $a$ and $b$ respectively.
They choose their response bits without any communication between them.
In particular, Alice and Bob do not know each other's secret bits or response bits.

## Win Condition
Alice and Bob win or lose depending on their secret bits $x,y$ and their response bits $a,b$.
They win if they give the same response to the referee, *except in the case where both Alice and Bob received $1$ as their secret bit*.
In that case, they win if they give *different* responses to the referee.
In every other case they lose.
This win condition is summarized by the formula $a\operatorname{XOR}b=x\operatorname{AND}y$.
Another way to represent this win condition is with a payoff table:

|(x,y) ↓ \\ (a,b) →|(0,0)|(0,1)|(1,0)|(1,1)|
|------------------|----:|----:|----:|----:|
|         **(0,0)**|    1|    0|    0|    1|
|         **(0,1)**|    1|    0|    0|    1|
|         **(1,0)**|    1|    0|    0|    1|
|         **(1,1)**|    0|    1|    1|    0|

This lists every combination of secret bits $(x,y)$ and responses $(a,b)$ along with the score the players receive.
Here an entry of 1 means the players win (score 1) and an entry of 0 means the players lose (score 0).

## Classical and Quantum strategies
What is the best strategy for Alice and Bob, recalling that they cannot talk to each other while the game is in play?
One idea is that they can plan beforehand that they will both always respond with 0.
Then they will win 3/4 of the time.
(Consult the first column of the table above.)
It turns out that a win probability of $\frac{3}{4}=0.75$ is the best Alice and Bob can do _without quantum resources_.

This changes if Alice and Bob share an entangled quantum state!
One can show that in that case it is possible for them to win with probability $\cos(\pi/8)^2=0.85\ldots$.
To see how this works, we will have to explain a bit about states and measurements in quantum theory.

### Measurements in Quantum Mechanics
A quantum measurement is represented mathematically by what's called a [POVM](http://en.wikipedia.org/wiki/Measurement_in_quantum_mechanics#Generalized_measurement_(POVM)).
In our case, a POVM is a set of Hermitian matrices $\{M_i\}_i$ (one for each possible outcome $i$ of the measurement) where the matrices are positive semidefinite (all eigenvalues nonnegative) and sum to the identity.
For example 
$$
\left\{
\begin{bmatrix}
1 & 0 & 0\\
0 & 0.1 & 0\\
0 & 0 & 0.3
\end{bmatrix},
\begin{bmatrix}
0 & 0 & 0\\
0 & 0.9 & 0\\
0 & 0 & 0.7
\end{bmatrix}
\right\}
$$
is a quantum measurement.
So is
$$
\left\{
\frac{1}{3}
\begin{bmatrix}
1 & 1\\
1 & 1
\end{bmatrix},
\frac{1}{3}
\begin{bmatrix}
1 & e^{-i2\pi/3}\\
e^{i2\pi/3} & 1
\end{bmatrix},
\frac{1}{3}
\begin{bmatrix}
1 & e^{i2\pi/3}\\
e^{-i2\pi/3} & 1
\end{bmatrix}
\right\}.
$$


The outcomes of measurements in quantum mechanics are stochastic, but the _probabilities_ for getting different measurement outcomes are determined by a principle called the Born Rule:
Say Alice has a quantum state with [density matrix](https://en.wikipedia.org/wiki/Density_matrix) $\rho$.
If she measures this quantum state using the POVM $\{M_1,M_2,\cdots,M_m\}$, the Born Rule says she will obtain outcome $i$ with probability $p_i=\operatorname{tr}(\rho M_i)$.

This rule extends to joint probabilities as well:
Say Alice and Bob share a quantum state $\rho_{AB}$.
If Alice measures her part of the state with a POVM $\{M_1,M_2,\cdots,M_m\}$ and Bob measures his part of the state with POVM $\{N_1,N_2,\cdots,N_n\}$, the probability of the joint outcome $(i,j)$ occurring is
$$
p_{ij}=\operatorname{tr}\big(\rho_{AB}\,(M_i\otimes N_j)\big).
$$
The symbol $\otimes$ represents a tensor product, which, for our purposes, is the same thing as the [Kronecker product](https://en.wikipedia.org/wiki/Kronecker_product) of matrices (implemented in numpy with `np.kron`).

### Winning CHSH Using Entanglement
If Alice and Bob share a quantum state $\rho_{AB}$ before the game, then there is a new class of strategies they can consider:
Alice chooses two POVMs beforehand, one for when her secret bit is 0 and one for when her secret bit is 1.
Bob does the same.
When playing the game, Alice and Bob implement the POVM measurement corresponding to their own secret bit and uses the measurement outcome as their response.
With appropriately chosen $\rho_{AB}$ and POVMs, Alice and Bob can win CHSH more often than the classical best of 3/4 of the time!

Let's get specific.
Say Alice and Bob share
$$
\rho_{AB}
=
\begin{bmatrix}
\frac{1}{2} & 0 & 0 & \frac{1}{2}\\
0 & 0 & 0 & 0\\
0 & 0 & 0 & 0\\
\frac{1}{2} & 0 & 0 & \frac{1}{2}\\
\end{bmatrix} ,
$$
Alice chooses POVMs
$$\left\{
\begin{bmatrix}
1 & 0\\
0 & 0
\end{bmatrix},
\begin{bmatrix}
0 & 0\\
0 & 1
\end{bmatrix}
\right\}$$ 
for when the secret bit is 0 and 
$$\left\{
\begin{bmatrix}
\frac{1}{2} & \frac{1}{2}\\
\frac{1}{2} & \frac{1}{2}
\end{bmatrix},
\begin{bmatrix}
\frac{1}{2} & -\frac{1}{2}\\
-\frac{1}{2} & \frac{1}{2}
\end{bmatrix}
\right\}$$
for when the secret bit is 1,
and Bob chooses POVMs
$$\left\{
\begin{bmatrix}
\cos^2(\frac{\pi}{8}) & \cos(\frac{\pi}{8})\sin(\frac{\pi}{8})\\
\cos(\frac{\pi}{8})\sin(\frac{\pi}{8}) & \sin^2(\frac{\pi}{8})
\end{bmatrix},
\begin{bmatrix}
\sin^2(\frac{\pi}{8}) & -\cos(\frac{\pi}{8})\sin(\frac{\pi}{8})\\
-\cos(\frac{\pi}{8})\sin(\frac{\pi}{8}) & \cos^2(\frac{\pi}{8})
\end{bmatrix}
\right\}$$ 
for secret 0 and 
$$\left\{
\begin{bmatrix}
\cos^2(\frac{\pi}{8}) & -\cos(\frac{\pi}{8})\sin(\frac{\pi}{8})\\
-\cos(\frac{\pi}{8})\sin(\frac{\pi}{8}) & \sin^2(\frac{\pi}{8})
\end{bmatrix},
\begin{bmatrix}
\sin^2(\frac{\pi}{8}) & \cos(\frac{\pi}{8})\sin(\frac{\pi}{8})\\
\cos(\frac{\pi}{8})\sin(\frac{\pi}{8}) & \cos^2(\frac{\pi}{8})
\end{bmatrix}
\right\}$$
for secret 1.
We can use the Born Rule to calculate the probabilities of the various actions Alice and Bob could take when implementing the above strategy.
Explicitly, the probability that Alice and Bob make the actions $(a, b)$ when they receive bits $(x, y)$ is
$$
p(a,b\mid x,y)
=
\operatorname{tr}\big(\rho_{AB}(M(a\mid x)\otimes N(b\mid y))\big)
$$
Where by $M(a\mid x)$ we mean the $a$-th matrix in Alice's POVM corresponding to secret bit value $x$.
Similarly $N(b\mid y)$ is the $b$-th matrix in whichever of Bob's POVMs corresponds to $y$.

With $p(a,b\mid x,y)$ calculated, you can get calculate the probability of winning with the strategy by
$$
\text{prob win}
=
\sum_{x,y,a,b}p(x,y)p(a,b\mid x,y)V(x,y,a,b)
$$

Let's check the win probability of the above strategy for the CHSH game using the cell below.




In [ ]:
# Construct Alice's POVMs
M = np.array([
    [[[1, 0],      # Alice's POVM for when x = 0
      [0, 0]],

     [[0, 0],
      [0, 1]]],


    [[[0.5, 0.5],  # Alice's POVM for when x = 1
      [0.5, 0.5]],

     [[0.5, -0.5],
      [-0.5, 0.5]]],
])


# Construct Bob's POVMs
c, s = np.cos(np.pi/8), np.sin(np.pi/8)
N = np.array([
    [[[c * c, c * s],    # Bob's POVM for when y = 0
      [s * c, s * s]],

     [[s * s, -c * s],
      [-s * c, c * c]]],


    [[[c * c, -s * c],   # Bob's POVM for when y = 1
      [-c * s, s * s]],
      
     [[s * s, s * c],
      [c * s, c * c]]],
])


# the shared entangled state
rho = np.array([
    [0.5, 0.0, 0.0, 0.5],
    [0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0],
    [0.5, 0.0, 0.0, 0.5],
])

# the CHSH payoff table, reshaped into a tensor V[x, y, a, b]
payoff = np.array([
    [1, 0, 0, 1],
    [1, 0, 0, 1],
    [1, 0, 0, 1],
    [0, 1, 1, 0],
]).reshape(2, 2, 2, 2)

# the referee draws (x, y) uniformly, so every pair has probability 1/4
p_xy = 1/4


num_questions, num_answers = M.shape[0], M.shape[1]
win_probability = 0.0
for x in range(num_questions):
    for y in range(num_questions):
        for a in range(num_answers):
            for b in range(num_answers):
                p_ab_given_xy = np.trace(rho @ np.kron(M[x, a], N[y, b]))
                win_probability += p_xy * payoff[x, y, a, b] * p_ab_given_xy

print(f"Win probability: {win_probability}")




## Nonlocal Games in General

CHSH is an example of what is called a **nonlocal game**. 
This is a cooperative game where a referee give **questions** to the players who respond with **answers**.
The players get a joint score based on their combined questions and answers.
The players must choose their answers without communicating with each other.

In general such a game is specified by two things:

* a payoff tensor `V[x, y, a, b]`, the score when the referee asks $(x,y)$ and the players answer $(a,b)$;
* a question distribution `pi[x, y]`, the probability that the referee draws the pair $(x,y)$.

A classical strategy is any plan Alice and Bob can implement without entanglement.
The best score Alice and Bob can get on a nonlocal game using just classical strategies is called the **classical value** of the game, denoted by $\omega_c$.
This classical value can always be achieved by some pair of deterministic rules $x\mapsto a_x$ and $y\mapsto b_y$, so one way to find the classical value of a nonlocal game is to enumerate over all possible deterministic rules and see what the best score achieved is.
We wrote a helper function `get_classical_best(V, pi)` that does this for you.

A quantum strategy, on the other hand, is a shared state $\rho$ together with one POVM per question for each player, as in CHSH above.
Its expected payoff is
$$
\sum_{x,y}\pi(x,y)\sum_{a,b} V(x,y,a,b)\,\operatorname{tr}\big(\rho\,(M^x_a\otimes N^y_b)\big),
$$
which `expected_payoff(V, pi, rho, M, N)` computes.
The **quantum value** $\omega^*$ is the best score possible over all quantum strategies.

It is always the case that $\omega^*\geq\omega_c$.
We say that a game has **quantum advantage** when $\omega^*>\omega_c$.

The problems in this challenge will involve finding quantum strategies that demonstrate a quantum advantage.




In [ ]:
pi = np.ones((2, 2))/4  # the referee draws (x, y) uniformly

print(f"CHSH classical value: {get_classical_best(payoff, pi)}")
print(f"Quantum strategy above: {expected_payoff(payoff, pi, rho, M, N)}")





# Solution Formatting Instructions

For most of the questions your answer will consist of a quantum state, POVMs, or both.

* What you submit have to be valid density matrices and POVMs.
You can use `is_valid_state` and `is_valid_povm` to check that they are.
Anything that fails these basic checks scores nothing.
Note that anything that passes these basic checks is then projected onto the exactly valid density matrix and POVMs nearest to it before being scored.
(This is to standardize things and guarantee that numerical differences can never be laundered into quantum advantage.)
See `check_validity.py` for the functions we will use to do this.

* Where $d$ is the dimension of Alice's and Bob's local quantum systems, a quantum state should be a $d^2\times d^2$ matrix.
Alice's POVMs should be turned in as a numpy array `M` with shape `(num_questions, num_answers, d, d)`, so that the first index runs over the question received, and `M[0]`, `M[1]`, `M[2]`, etc. are each a POVM where matrices `M[x, 0]`, `M[x, 1]`, `M[x, 2]`, etc. are $d\times d$ matrices corresponding to measurement outcomes 0, 1, 2, ... respectively.
Bob's POVMs should similarly be turned in as a numpy array `N`.

* In your answers, Alice's and Bob's local dimension `d` is allowed to be anything up to 5 (except in Problem 1 where the state is given to you and `d` should accordingly be 2).
This restriction makes the grading easier for us.

* Every matrix you submit must be `complex128`.

Here is a chart of some of the objects and conventions we'll use, including some of the conventions your solutions must adhere to:
| object | shape | notes |
|---|---|---|
| `V` | `(num_questions, num_questions, num_answers, num_answers)` | indexed `[x, y, a, b]` |
| `pi` | `(num_questions, num_questions)` | probability distribution, indexed `[x, y]` |
| `rho` | `(d*d, d*d)` | density matrix. Alice's subsystem first |
| `M`, `N` | `(num_questions, num_answers, d, d)` | indexed `[x, a]` and `[y, b]` |








# Problem 1: CHSH With a Different State

Suppose Alice and Bob now share the state
$$
\rho_{AB}
=
\begin{bmatrix}
1/2 & 0 & 0 & -1/2\\
0 & 0 & 0 & 0\\
0 & 0 & 0 & 0\\
-1/2 & 0 & 0 & 1/2\\
\end{bmatrix},
$$
this differs from the warmup state only in the sign of its off-diagonal entries.
The warmup POVMs are no longer a good strategy for this state.

Find POVMs for Alice and for Bob that still win the CHSH game more than 75% of the time.
Each should be an array of shape `(2, 2, 2, 2)`, indexed `[x, a]` for Alice and `[y, b]` for Bob.

**Submit:** `problem_1.npz` with keys `M`, `N`.




In [ ]:
# Put your answers in these two lines
M = ...  # your POVMs for Alice, shape (2, 2, 2, 2), indexed [x, a]
N = ...  # your POVMs for Bob, shape (2, 2, 2, 2), indexed [y, b]


# Check your answers
rho = np.array([
    [0.5, 0.0, 0.0, -0.5],
    [0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0],
    [-0.5, 0.0, 0.0, 0.5],
])

print(f"M contains valid POVMs: {is_valid_povm(M)}")
print(f"N contains valid POVMs: {is_valid_povm(N)}")
print(f"Win probability: {expected_payoff(payoff, pi, rho, M, N)}")

np.savez(
    'problem_1.npz',
    M=np.asarray(M, dtype=complex),
    N=np.asarray(N, dtype=complex),
)




# Problem 2: A Trading Game

[Recent research](https://arxiv.org/abs/2407.21723) makes the case that quantum entanglement could be useful for high-frequency trading.

The two major North American stock exchanges are the Nasdaq Stock Exchange and the New York Stock Exchange (NYSE).
These exchanges, and many of the trading algorithms that interact with them, are run from two respective data centers in New Jersey that are about 55 km apart.
Because of this geographical separation and the finite speed of light, it takes at least 180 $\mu\text{s}$ to send any kind of communication from one exchange to the other.

Suppose you are a trader with trading algorithms running at both Nasdaq and NYSE.
Often in high-frequency trading, the opportunities you must act on are fleeting, on a timescale much less than even 180 $\mu\text{s}$.
This means that your algorithms must react to market signals before there is any time for them to communicate with your other algorithms at different locations.
This raises the risk that your algorithms could end up acting at cross purposes.

This situation of cooperating agents that must act without communicating has the essential form of a nonlocal game, and as we've seen, entanglement can sometimes help players score better in nonlocal games.
The upshot for traders is that, depending on the exact trading scenario, shared entanglement between your trading algos could help them make better decisions!

Below is the payoff table of a game inspired by a high-frequency trading situation:

|(x,y) ↓ \\ (a,b) →|(0,0)|(0,1)|(1,0)|(1,1)|
|------------------|----:|----:|----:|----:|
|         **(0,0)**|    1|    0|    0|    0|
|         **(0,1)**|    0|    0|    1|    0|
|         **(1,0)**|    0|    1|    0|    0|
|         **(1,1)**|    0|    1|    1|    0|

We won't give the details, but here $x,y$ represent certain market signals and $a,b$ represent the decision to either keep a trade order open or to cancel it.
Assume the four joint signals are equally likely, so $\pi(x,y)=1/4$.

Find a quantum strategy that beats every classical strategy at this game, and make its advantage as large as you can.

**Submit:** `problem_2.npz` with keys `rho`, `M`, `N`.




In [ ]:
V_trading = np.array([
    [1, 0, 0, 0],
    [0, 0, 1, 0],
    [0, 1, 0, 0],
    [0, 1, 1, 0],
]).reshape(2, 2, 2, 2)

print(f"classical value: {get_classical_best(V_trading, pi)}")

rho = ...  # your shared state,    shape (d*d, d*d) with d <= 5
M = ...    # your POVMs for Alice, shape (2, 2, d, d), indexed [x, a]
N = ...    # your POVMs for Bob,   shape (2, 2, d, d), indexed [y, b]

print(f"your strategy: {expected_payoff(V_trading, pi, rho, M, N)}")

np.savez(
    'problem_2.npz',
    rho=np.asarray(rho, dtype=complex),
    M=np.asarray(M, dtype=complex),
    N=np.asarray(N, dtype=complex),
)




# Problem 3: How Much Can Entanglement Buy?

In CHSH entanglement is worth $\cos^2(\pi/8)-3/4\approx 0.1036$ of expected payoff.
In the trading game of Problem 2 it is worth less.
How much is the most it can ever be worth, for a game this small?

Consider every nonlocal game with 2 questions and 2 answers per player, uniform questions $\pi(x,y)=1/4$, and payoffs that are all either 0 or 1.
Each such game is a tensor `V[x, y, a, b]` of sixteen bits, so there are $2^{16}=65536$ of them.

Find the one with the largest quantum advantage
$$
\omega^*(V) - \omega_c(V)
$$
and a quantum strategy that achieves it.
Your score is the advantage we verify from the strategy you submit, so a near-optimal strategy for the right game still scores well.

**Submit:** `problem_3.npz` with keys `V`, `rho`, `M`, `N`.




In [ ]:
V = ...    # the game you found,     shape (2, 2, 2, 2), indexed [x, y, a, b],  entries 0 or 1
rho = ...  # your shared state,      shape (d*d, d*d) with d <= 5
M = ...    # your POVMs for Alice,   shape (2, 2, d, d), indexed [x, a]
N = ...    # your POVMs for Bob,     shape (2, 2, d, d), indexed [y, b]

advantage = expected_payoff(V, pi, rho, M, N) - get_classical_best(V, pi)
print(f"advantage: {advantage}")

np.savez(
    'problem_3.npz',
    V=np.asarray(V, dtype=np.int64),
    rho=np.asarray(rho, dtype=complex),
    M=np.asarray(M, dtype=complex),
    N=np.asarray(N, dtype=complex),
)





# Problem 4: How Often Does Entanglement Help?

Not every decision problem benefits from entanglement.
Plenty of nonlocal games have no entangled strategy that beats the best classical one at all.
So if we are interested in applying entanglement to real decision-making problems a natural question is *how often does entanglement actually help?*
In this problem you will find a quantitative answer to a simple version of this problem.

Take the same 65536 games as Problem 3, namely games with 2 questions and 2 answers per player, uniformly chosen questions, and payoffs 0 or 1.
**How many of these 65536 games have quantum advantage?**

Count a game as having quantum advantage if $\omega^*(V) > \omega_c(V) + 10^{-6}$.
Your answer is an integer between 0 and 65536, and you are scored on how close it is to the true count.

**Submit:** `problem_4.npz` with key `num_games_with_quantum_advantage` whose value is your integer answer.




In [ ]:
np.savez(
    'problem_4.npz',
    num_games_with_quantum_advantage=...,  # your integer answer here
)





# Problem 5: Random Nonlocal Games

Problem 4 answers "how often does entanglement help?" exactly, but only for the very smallest games.
Here we ask the same question for games large enough that counting is hopeless.

Consider two-player games with **10 questions** and **2 answers** per player, uniform questions, and payoffs that are all 0 or 1.
There are $2^{10\cdot 10\cdot 2\cdot 2}\approx 2.582\times 10^{120}$ of them.
We cannot count the ones with quantum advantage, but we can estimate a lower bound on what *fraction* of them have it: take games at random, prove advantage for as many as you can, and calculate a confidence bound.

For this problem we are providing you with a _fixed sequence of random games_.
The function `games.get_game(i)` returns game number `i` of a fixed public sequence of random games, and `games.classical_values()` returns our exact classical values for them.

You will pick a number $N$, work through games $0,1,\dots,N-1$, and submit a witness of quantum advantage (i.e. the shared state and POVMs of the outperforming quantum strategy) for every one where you can beat the classical best.

## Rules and things to know

For game `i`, a witness is a quantum strategy $(\rho, M, N)$ such that

* the local dimension is at most $d=5$, so $\rho$ is at most $25\times 25$ and each POVM element at most $5\times 5$;
* $\rho$ is a density matrix and $M$, $N$ are valid POVMs;
* its expected payoff is at least $\omega_c + 10^{-9}$, where $\omega_c$ is *our* published classical value for game `i`.

A few things to know before you start:

* The classical values we publish are the single source of truth. They are exact, so there is nothing to disagree about. 
The $10^{-9}$ margin is to avoid counting numerical error as quantum advantage.
* Your strategies are repaired to ensure they are valid density matrices and POVMs to numerical precision before scoring, as described in `check_validity.py`. 
Repair never meaningfully changes a valid strategy, but we do it to ensure that numerical errors can never be compounded into a spurious quantum advantage.
* The dimension cap is part of the question. 
You are bounding the probability that a random game has advantage *at dimension 5 or below*.
* **You will choose how many games you take on.**
Choose $N$ from the set $\{10, 100, 1000, 10000, 100000\}$. 
You will try to find quantum advantage for as many games as you can in the first $N$ games in the sequence.
* For ease of grading, you may submit at most $25000$ witnesses. 
The function `games.save_witnesses` writes the file and pads strategies of dimension below 5 up to 5 for you.

## Scoring

If you submit $k$ verified witnesses out of the $N$ games you took on, your score is
$$
\operatorname{score}(k, N)
=
\operatorname{Beta}^{-1}(0.05;\, k,\, N-k+1).
$$
(This formula gives a 95% lower confidence bound on the fraction of random games with quantum advantage.)
For your convenience, we have provided a function `games.score(k, N)` which calculates this for you.

Your score rises with $k$, so every witness you find helps, and for a fixed success rate $k/N$, it rises with $N$, so for a sound method, taking on more games should help too.
It can never exceed the true rate, and any witness you miss only lowers $k$, so whatever you hand in is a conservative lower bound.
Choosing a large $N$ and then failing to crack most of those games will hurt you, so the game is to push $N$ as far as you can still cover it well.

Run the next cell to see a plot that might help you understand the trade-off.






In [ ]:
N_choices = [10, 100, 1000, 10000, 100000]
for rate in (0.05, 0.10, 0.15, 0.20):
    plt.semilogx(N_choices, [score(round(rate * N), N) for N in N_choices], 'o-', label=f'{rate:.0%} success rate')
    plt.axhline(rate, color='k', lw=0.5, ls=':')
plt.xlabel('N (games taken on)')
plt.ylabel('score')
plt.title('Dotted lines are the true rate, which the score approaches from below')
plt.legend()
plt.grid(alpha=0.3)
plt.show()





Here is everything you need to get started.
Set `num_games` to the $N$ you are taking on, and fill in `witnesses` with one entry per game you can beat.

The games come from a published seed via SHA-256, so they do not depend on your NumPy version or on the order you generate them in.
The fingerprint printed below should read `cb33fab53cfd870b`; if it does not, something is wrong with your checkout.

**Submit:** `problem_5.npz`, written by `save_witnesses`.
If it is under 100 MB, commit it like the other files.
If it is larger, git will reject it, so attach it to a GitHub Release named `submission` instead, as described in the README.







In [ ]:
num_games = 1000 # how many games you are taking on, a power of 10 up to MAX_GAMES

omega_c = classical_values(num_games)
pi_10 = np.ones((10, 10))/100 # the referee draws (x, y) uniformly

print(f"fingerprint of your copy of the games: {fingerprint()}")
print(f"classical values: mean {omega_c.mean():.4f}, range [{omega_c.min()}, {omega_c.max()}]")
print(f"game 0 has payoff tensor of shape {get_game(0).shape} and classical value {omega_c[0]}")

witnesses = {} # game index -> (rho, M, N)

for i in range(num_games):
    V = get_game(i).astype(float)
    target = omega_c[i] + TOLERANCE
    # ... find a strategy (rho, M, N) whose expected_payoff(V, pi_10, rho, M, N) reaches
    # target, and if you find one, record it as witnesses[i] = (rho, M, N) ...

print(f"\n{len(witnesses)} witnesses, score {score(len(witnesses), num_games):.6f}")
save_witnesses('problem_5.npz', witnesses, num_games)
